In [1]:
MODEL_NAME  = 'sentence-transformers/all-mpnet-base-v2'
MODEL_ID    = 'mpnet_base'
MODEL_LABEL = 'all-mpnet-base-v2'
RESULTS_DIR = '/home/arshad/Network-project/data/embedding_results/'
DATA_DIR    = '/home/arshad/Network-project/data/'

print(f"🤖 Model:  {MODEL_LABEL}")
print(f"📁 Output: {RESULTS_DIR}{MODEL_ID}_results.csv")

🤖 Model:  all-mpnet-base-v2
📁 Output: /home/arshad/Network-project/data/embedding_results/mpnet_base_results.csv


In [2]:
import gc, os, json, pickle, time, warnings
import numpy as np
import pandas as pd
import faiss
import psutil
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
tqdm.pandas()
warnings.filterwarnings('ignore')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load dataset
df = pd.read_csv(DATA_DIR + 'final_risk_scored.csv', low_memory=False)
df['MITRE_Tactic']    = df['MITRE_Tactic'].fillna('None')
df['MITRE_Technique'] = df['MITRE_Technique'].fillna('None')

# Load MITRE techniques
with open(DATA_DIR + 'mitre_techniques.pkl', 'rb') as f:
    techniques = pickle.load(f)

# Load tactic classifier
with open(DATA_DIR + 'tactic_classifier.pkl', 'rb') as f:
    clf_data = pickle.load(f)
clf          = clf_data['model']
le           = clf_data['encoder']
feature_cols = clf_data['features']

# Attack samples only
attack_df = df[df['Label'] != 'BENIGN'].copy()

print(f"✅ Dataset loaded: {len(attack_df):,} attack samples")
print(f"✅ MITRE techniques: {len(techniques)}")
print(f"💾 RAM: {psutil.virtual_memory().available/1024**3:.1f} GB")

✅ Dataset loaded: 57,143 attack samples
✅ MITRE techniques: 691
💾 RAM: 7.8 GB


In [3]:
def get_parent_id(tid):
    return tid.split('.')[0] if isinstance(tid, str) and '.' in tid else tid

tactic_map = {
    'BENIGN':                    'benign',
    'DoS Hulk':                  'impact',
    'DoS GoldenEye':             'impact',
    'DoS slowloris':             'impact',
    'DoS Slowhttptest':          'impact',
    'DDoS':                      'impact',
    'PortScan':                  'discovery',
    'FTP-Patator':               'credential-access',
    'SSH-Patator':               'credential-access',
    'Bot':                       'command-and-control',
    'Web Attack - Brute Force':  'initial-access',
    'Web Attack - XSS':          'initial-access',
    'Web Attack - SQL Injection':'initial-access',
    'Heartbleed':                'initial-access',
    'Infiltration':              'lateral-movement',
}

attack_df['true_tactic']  = attack_df['Label'].map(tactic_map)
attack_df['true_parent']  = attack_df['MITRE_Technique'].apply(get_parent_id)

print("✅ Helper functions ready!")

✅ Helper functions ready!


In [4]:
import os

# Get model size before loading
cache_dir = os.path.expanduser('~/.cache/huggingface/hub')

print(f"⏳ Loading model: {MODEL_LABEL}")
t_start    = time.time()
emb_model  = SentenceTransformer(MODEL_NAME)
t_load     = time.time() - t_start
print(f"✅ Model loaded in {t_load:.1f}s")

# Get embedding dimension
test_emb = emb_model.encode(["test"], convert_to_numpy=True)
emb_dim  = test_emb.shape[1]
print(f"✅ Embedding dimension: {emb_dim}")

# Embed MITRE techniques
technique_texts = [
    f"Technique {t['technique_id']}: {t['technique_name']}. "
    f"Tactics: {t['tactics']}. "
    f"Description: {t['description'][:500]}"
    for t in techniques
]

print(f"\n⏳ Embedding {len(techniques)} MITRE techniques...")
t_start = time.time()
tech_embeddings = emb_model.encode(
    technique_texts, batch_size=64,
    show_progress_bar=True, convert_to_numpy=True
)
t_tech = time.time() - t_start
faiss.normalize_L2(tech_embeddings)

# Build FAISS index
index = faiss.IndexFlatIP(emb_dim)
index.add(tech_embeddings)

print(f"✅ Techniques embedded in {t_tech:.1f}s")
print(f"✅ FAISS index: {index.ntotal} techniques")
print(f"💾 RAM: {psutil.virtual_memory().available/1024**3:.1f} GB")

⏳ Loading model: all-mpnet-base-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded in 21.4s
✅ Embedding dimension: 768

⏳ Embedding 691 MITRE techniques...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

✅ Techniques embedded in 5.0s
✅ FAISS index: 691 techniques
💾 RAM: 7.4 GB


In [5]:
print(f"⏳ Embedding {len(attack_df):,} alert texts...")
t_start = time.time()
alert_embeddings = emb_model.encode(
    attack_df['alert_text'].tolist(),
    batch_size=256, show_progress_bar=True,
    convert_to_numpy=True
)
t_alert = time.time() - t_start
faiss.normalize_L2(alert_embeddings)

print(f"✅ Alert texts embedded in {t_alert:.1f}s")
print(f"✅ Shape: {alert_embeddings.shape}")
print(f"💾 RAM: {psutil.virtual_memory().available/1024**3:.1f} GB")

# Total embedding time
total_embed_time = t_tech + t_alert
print(f"\n⏱️  Total embedding time: {total_embed_time:.1f}s")

⏳ Embedding 57,143 alert texts...


Batches:   0%|          | 0/224 [00:00<?, ?it/s]

✅ Alert texts embedded in 122.0s
✅ Shape: (57143, 768)
💾 RAM: 6.9 GB

⏱️  Total embedding time: 127.0s


In [6]:
print("⏳ Predicting tactics with ML classifier...")
X_all        = attack_df[feature_cols].replace(
                   [np.inf, -np.inf], np.nan).fillna(0)
pred_tactics = le.inverse_transform(clf.predict(X_all))
attack_df['pred_tactic'] = pred_tactics
print(f"✅ Tactic predictions done!")

⏳ Predicting tactics with ML classifier...
✅ Tactic predictions done!


In [7]:
print(f"⏳ Running hybrid MITRE matching with {MODEL_LABEL}...")
t_start = time.time()

pred1_ids, pred2_ids, pred3_ids = [], [], []

for i in tqdm(range(len(attack_df)), desc="Matching"):
    tactic   = pred_tactics[i]
    tac_idxs = [j for j,t in enumerate(techniques)
                if tactic in t['tactics']]

    if len(tac_idxs) >= 3:
        sub_embs  = tech_embeddings[tac_idxs]
        sims      = (alert_embeddings[i:i+1] @ sub_embs.T)[0]
        top3_loc  = np.argsort(sims)[::-1][:3]
        top3_idx  = [tac_idxs[k] for k in top3_loc]
    else:
        _, idx   = index.search(alert_embeddings[i:i+1], 3)
        top3_idx = idx[0].tolist()

    pred1_ids.append(get_parent_id(techniques[top3_idx[0]]['technique_id']))
    pred2_ids.append(get_parent_id(techniques[top3_idx[1]]['technique_id']))
    pred3_ids.append(get_parent_id(techniques[top3_idx[2]]['technique_id']))

t_match = time.time() - t_start
attack_df['pred1'] = pred1_ids
attack_df['pred2'] = pred2_ids
attack_df['pred3'] = pred3_ids

print(f"✅ Matching done in {t_match:.1f}s")
gc.collect()

⏳ Running hybrid MITRE matching with all-mpnet-base-v2...


Matching: 100%|████████████████████████| 57143/57143 [00:03<00:00, 18715.24it/s]


✅ Matching done in 3.1s


46

In [8]:
from sklearn.metrics import accuracy_score

# Overall accuracy
top1 = (attack_df['true_parent'] == attack_df['pred1']).mean()
top3 = attack_df.apply(
    lambda r: r['true_parent'] in [r['pred1'], r['pred2'], r['pred3']],
    axis=1
).mean()

# Tactic accuracy
tactic_acc = accuracy_score(
    attack_df['true_tactic'],
    attack_df['pred_tactic']
)

print(f"📊 RESULTS — {MODEL_LABEL}")
print("=" * 55)
print(f"  Tactic Accuracy:          {tactic_acc:.2%}")
print(f"  MITRE Top-1 Accuracy:     {top1:.2%}")
print(f"  MITRE Top-3 Accuracy:     {top3:.2%}")
print(f"  Embedding Dimension:      {emb_dim}")
print(f"  Total Embedding Time:     {total_embed_time:.1f}s")
print(f"  Matching Time:            {t_match:.1f}s")

print(f"\n📊 Per Attack Type:")
print("-" * 55)
print(f"{'Attack Type':<35} {'Top-1':>6}  {'Top-3':>6}")
print("-" * 55)
for label in sorted(attack_df['Label'].unique()):
    sub    = attack_df[attack_df['Label'] == label]
    t1     = (sub['true_parent'] == sub['pred1']).mean()
    t3     = sub.apply(lambda r: r['true_parent'] in [
                 r['pred1'], r['pred2'], r['pred3']], axis=1).mean()
    status = "✅" if t3 >= 0.5 else "⚠️ "
    print(f"  {status} {label:<33} {t1:>5.0%}   {t3:>5.0%}")

# Save results
per_attack = {}
for label in attack_df['Label'].unique():
    sub = attack_df[attack_df['Label'] == label]
    per_attack[label] = {
        'top1': (sub['true_parent'] == sub['pred1']).mean(),
        'top3': sub.apply(lambda r: r['true_parent'] in [
                    r['pred1'], r['pred2'], r['pred3']], axis=1).mean()
    }

results = {
    'model_id':          MODEL_ID,
    'model_label':       MODEL_LABEL,
    'emb_dimension':     emb_dim,
    'tactic_accuracy':   round(tactic_acc, 4),
    'top1_accuracy':     round(top1, 4),
    'top3_accuracy':     round(top3, 4),
    'embed_time_sec':    round(total_embed_time, 1),
    'match_time_sec':    round(t_match, 1),
    'total_time_sec':    round(total_embed_time + t_match, 1),
}

# Add per-attack results
for label, scores in per_attack.items():
    safe_label = label.replace(' ', '_').replace('-', '_')
    results[f'{safe_label}_top1'] = round(scores['top1'], 4)
    results[f'{safe_label}_top3'] = round(scores['top3'], 4)

results_df = pd.DataFrame([results])
results_df.to_csv(
    f"{RESULTS_DIR}{MODEL_ID}_results.csv",
    index=False
)

print(f"\n✅ Results saved → {RESULTS_DIR}{MODEL_ID}_results.csv")
print(f"\n🎉 {MODEL_LABEL} evaluation complete!")

📊 RESULTS — all-mpnet-base-v2
  Tactic Accuracy:          99.81%
  MITRE Top-1 Accuracy:     59.51%
  MITRE Top-3 Accuracy:     91.01%
  Embedding Dimension:      768
  Total Embedding Time:     127.0s
  Matching Time:            3.1s

📊 Per Attack Type:
-------------------------------------------------------
Attack Type                          Top-1   Top-3
-------------------------------------------------------
  ⚠️  Bot                                 35%     39%
  ✅ DDoS                                46%    100%
  ✅ DoS GoldenEye                       72%    100%
  ✅ DoS Hulk                            43%    100%
  ✅ DoS Slowhttptest                    83%    100%
  ✅ DoS slowloris                       67%    100%
  ✅ FTP-Patator                         68%     68%
  ⚠️  Heartbleed                           0%      0%
  ⚠️  Infiltration                         3%     11%
  ✅ PortScan                            98%     99%
  ✅ SSH-Patator                          0%     93%
  ⚠️